<a href="https://colab.research.google.com/github/maoyuqing50-code/Wiki-Trends-Analysis/blob/main/wikipedia_analysis_v4_DE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wikipedia DE — Linguistische Analyse (v4)
### IS-618

**What this notebook does:**
1. Loads German-language contested/stable article pairs (`final_pairs_de_strict.json`)
2. Extracts **7 linguistic features** across Layer 1 (F1–F2), Layer 2 (F3–F5), Layer 3 (F6–F7)
3. Runs **Wilcoxon signed-rank tests** on matched pairs with Holm correction
4. Runs **logistic regression** with topic controls
5. Produces **Bootstrap 95% CI**, VIF check, coefficient plot

**Feature map:**
| ID | Feature | Layer | Direction |
|----|---------|-------|----------|
| F1 | Epistemic hedge density | L1 | C > S |
| F2 | Affective stance marker density | L1 | C > S |
| F3 | Weasel word density | L2 | C > S |
| F4 | Attribution verb bias ratio | L2 | C > S |
| F5 | Lexical presupposition density (LLM) | L2 | C > S |
| F6 | Contrastive transition density | L3 | S > C |
| F7 | Lexical diversity (MTLD) | L3 | C > S |

**Files needed:**
- `final_pairs_de_strict.json` (placeholder — not yet available)


## 0. Install Dependencies

In [ ]:
!pip install spacy lexicalrichness statsmodels transformers torch seaborn -q
!python -m spacy download de_core_news_sm -q
print('Installation complete')

## 0b. Mount Google Drive & Set Checkpoint Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/wikipedia_de_v4_checkpoints/'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

F5_CHECKPOINT = CHECKPOINT_DIR + 'f5_entailment_scores_de.csv'
print(f'Checkpoint directory: {CHECKPOINT_DIR}')

## 1. Imports

In [ ]:
import json, re, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import scipy.stats as stats
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.utils import resample
from lexicalrichness import LexicalRichness
import torch
from transformers import pipeline
warnings.filterwarnings('ignore')
print('Imports OK')

## ⚠️ Before Running
Upload `final_pairs_de_strict.json` to this Colab session when available.
Drag-and-drop to the Files panel on the left, or run:
```python
from google.colab import files
files.upload()
```
Expected JSON structure (same as EN version):
- List of pair objects with keys: `contested`, `stable`, `score`, `same_topic`
- Each article dict: `title`, `label`, `clean_text`, `word_count`, `topic`, `age_days`, ...
- `label`: contested = 0, stable = 1
- Use `clean_text` for all feature extraction


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DATA LOADING — PLATZHALTER
# German pairs file not yet available. Replace this cell when
# final_pairs_de_strict.json is ready.
# ══════════════════════════════════════════════════════════════════════════════

# TODO: replace with actual file when available
# with open('final_pairs_de_strict.json', encoding='utf-8') as f:
#     pairs = json.load(f)

# TEMPORARY: create empty dataframe to allow rest of notebook to load
pairs = []
df = pd.DataFrame(columns=['title', 'label', 'text', 'topic', 'word_count', 'age_days', 'pair_id'])
print('⚠️  German pairs file not yet loaded — using empty dataframe placeholder')
print('    Replace this cell with actual data loading when final_pairs_de_strict.json is available')

### Data Loading Cell (activate when `final_pairs_de_strict.json` is available)

In [ ]:
# ── ACTIVATE when final_pairs_de_strict.json is available ───────────────────────
# Comment out the placeholder cell above and run this cell instead

# with open('final_pairs_de_strict.json', encoding='utf-8') as f:
#     pairs = json.load(f)
#
# records = []
# for p in pairs:
#     for role, art in [('contested', p['contested']), ('stable', p['stable'])]:
#         records.append({
#             'title'     : art['title'],
#             'label'     : art['label'],
#             'text'      : art['clean_text'],
#             'topic'     : art['topic'],
#             'word_count': art['word_count'],
#             'age_days'  : art['age_days'],
#             'pair_id'   : p['contested']['title'],
#         })
# df = pd.DataFrame(records)
# print(f'Total articles : {len(df)}')
# print(f'Labels         : {df.label.value_counts().to_dict()}')
# print(f'Topics         : {df.topic.value_counts().to_dict()}')

print('(Datei-Lade-Zelle — noch auskommentiert)')

## 2. spaCy Pipeline (German)

In [ ]:
import spacy
nlp = spacy.load('de_core_news_sm', disable=['ner'])
# de_core_news_sm already includes 'senter' for sentence segmentation
# do NOT add sentencizer — it conflicts with the built-in senter
print('spaCy de_core_news_sm loaded')
print(f'Pipeline components: {nlp.pipe_names}')

## 3. Word Lists & Helper Functions

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WORD LISTS
# ══════════════════════════════════════════════════════════════════════════════

# ── F1: Epistemic Hedge Density ────────────────────────────────────────────────
# Source (lexical): Helbig & Buscha (1991, p. 507) Modalwörter Class 1 + Class 6
# Source (morphological): spaCy MORPH Mood=Sub+Tense=Past (Konjunktiv II)
# Diewald (1999, pp. 17–18): sollte/wollte excluded as Quotativa

HEDGES_DE = [
    # Helbig & Buscha (1991, p.507) Klasse 1
    'anscheinend', 'möglicherweise', 'mutmaßlich', 'scheinbar',
    'schwerlich', 'vermutlich', 'vielleicht', 'wahrscheinlich',
    'wohl', 'womöglich',
    # Helbig & Buscha (1991, p.507) Klasse 6
    'angeblich', 'vorgeblich',
    # Additional epistemic adverbs
    'offenbar',  # borderline: can be epistemic; retained with caveat
    # Epistemic verbs
    'scheinen', 'erscheinen', 'vermuten', 'annehmen', 'glauben',
    'zweifeln', 'bezweifeln',
    # Probability adjectives
    'möglich', 'unwahrscheinlich', 'fraglich',
    'zweifelhaft', 'unsicher', 'unklar',
    # Perspective markers
    'meiner Meinung nach', 'meines Erachtens', 'nach meiner Ansicht',
    'nach meiner Einschätzung', 'soweit ich weiß',
]

# Konjunktiv II morphological hedge — detected via spaCy MORPH tags
# Covers: könnte, müsste, dürfte, möchte (modal verbs in Konjunktiv II)
# Implementation: supplement lexical count with morphological count in f1_hedge_density()
KONJUNKTIV_II_MODALS = {'können', 'müssen', 'dürfen', 'mögen'}
# Note: sollte (sollen) and wollte (wollen) excluded per Diewald (1999, pp. 17–18)
# as Quotativa — do not express epistemic speaker stance

In [ ]:
import urllib.request, zipfile, os

# ── F2: Affective Stance Marker Density ───────────────────────────────────────
# Source 1: Hyland (2018, pp. 268–269) Appendix Attitude Markers
#           German functional equivalents; translation justified by
#           cross-linguistic stability of metadiscourse categories
#           (Hyland 2018, p. 5; Kranich 2016)
#           'wesentlich' excluded — not in Hyland attitude markers list
#           'essentially' (EN) → 'im Wesentlichen' excluded to avoid
#           overlap with F1 epistemic hedges
# Source 2: SentiWS v2.0 high-intensity subset (|weight| >= 0.5)
#           Remus et al. (2010); wortschatz-leipzig.de/en/download/
#           License: CC BY-NC-SA 4.0
#           Limitation: SentiWS built on product reviews —
#           domain mismatch with Wikipedia acknowledged

HYLAND_ATTITUDE_DE = [
    # admittedly → einräumen, zugeben
    'einräumen', 'zugeben',
    # agree/disagree → zustimmen, ablehnen, widersprechen
    'zustimmen', 'ablehnen', 'widersprechen',
    # amazed/amazing/amazingly
    'erstaunt', 'erstaunlich', 'erstaunlicherweise',
    # appropriate/appropriately → angemessen, angemessenerweise
    'angemessen', 'angemessenerweise',
    # inappropriate/inappropriately
    'unangemessen', 'unangemessenerweise',
    # correctly → richtigerweise
    'richtigerweise',
    # curious/curiously → merkwürdig, merkwürdigerweise
    'merkwürdig', 'merkwürdigerweise',
    # desirable/desirably → wünschenswert, wünschenswerterweise
    'wünschenswert', 'wünschenswerterweise',
    # disappointed/disappointing/disappointingly
    'enttäuscht', 'enttäuschend', 'enttäuschenderweise',
    # dramatic/dramatically → dramatisch, dramatischerweise
    'dramatisch', 'dramatischerweise',
    # essential → unerlässlich (wesentlich excluded — overlaps F1)
    'unerlässlich',
    # expected/expectedly → zu erwarten, erwartungsgemäß
    'zu erwarten', 'erwartungsgemäß',
    # fortunate/fortunately → glücklicherweise, zum Glück
    'glücklicherweise', 'zum Glück',
    # hopeful/hopefully → hoffnungsvoll, hoffentlich
    'hoffnungsvoll', 'hoffentlich',
    # important/importantly → wichtig
    # Note: 'wichtigerweise' does not exist in German; adverbial
    # use covered by 'wichtig' as predicative adjective
    'wichtig',
    # interesting/interestingly → interessant, interessanterweise
    'interessant', 'interessanterweise',
    # prefer/preferable/preferably → bevorzugen, vorzuziehen, vorzugsweise
    'bevorzugen', 'vorzuziehen', 'vorzugsweise',
    # regret → bedauern
    'bedauern',
    # remarkable/remarkably → bemerkenswert, bemerkenswerterweise
    'bemerkenswert', 'bemerkenswerterweise',
    # shocked/shocking/shockingly
    'schockiert', 'schockierend', 'schockierenderweise',
    # striking/strikingly → auffallend, auffällig, auffallenderweise
    'auffallend', 'auffällig', 'auffallenderweise',
    # surprised/surprising/surprisingly
    'überrascht', 'überraschend', 'überraschenderweise',
    # unbelievable/unbelievably → unglaublich
    'unglaublich',
    # understandable/understandably → verständlich, verständlicherweise
    'verständlich', 'verständlicherweise',
    # unexpected/unexpectedly → unerwartet, unerwarteterweise
    'unerwartet', 'unerwarteterweise',
    # unfortunate/unfortunately → unglücklich, unglücklicherweise, bedauerlicherweise
    'unglücklich', 'unglücklicherweise', 'bedauerlicherweise',
    # unusual/unusually → ungewöhnlich, ungewöhnlicherweise
    'ungewöhnlich', 'ungewöhnlicherweise',
    # usual → gewöhnlich
    'gewöhnlich',
]

# ── Load SentiWS v2.0 high-intensity subset ───────────────────────────────────
SENTIWS_URL = 'http://pcai056.informatik.uni-leipzig.de/downloads/etc/SentiWS/SentiWS_v2.0.zip'
SENTIWS_ZIP = 'SentiWS_v2.0.zip'

if not os.path.exists('SentiWS_v2.0_Positive.txt'):
    print('Downloading SentiWS v2.0...')
    urllib.request.urlretrieve(SENTIWS_URL, SENTIWS_ZIP)
    with zipfile.ZipFile(SENTIWS_ZIP, 'r') as z:
        z.extractall('.')
    print('SentiWS v2.0 downloaded and extracted')
else:
    print('SentiWS v2.0 already present')

def parse_sentiws(filepath, threshold=0.5):
    """
    Extract lemmas and inflected forms from SentiWS file
    where |weight| >= threshold.
    Format: <Word>|<POS>\t<weight>\t<infl1>,<infl2>,...
    """
    words = []
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 2:
                continue
            lemma  = parts[0].split('|')[0].lower()
            weight = float(parts[1])
            if abs(weight) >= threshold:
                words.append(lemma)
                if len(parts) >= 3 and parts[2]:
                    for inf in parts[2].split(','):
                        words.append(inf.strip().lower())
    return words

SENTIWS_HIGH = list(set(
    parse_sentiws('SentiWS_v2.0_Positive.txt') +
    parse_sentiws('SentiWS_v2.0_Negative.txt')
))

# ── Combine and deduplicate ────────────────────────────────────────────────────
AFFECTIVE_MARKERS_DE = list(dict.fromkeys(HYLAND_ATTITUDE_DE + SENTIWS_HIGH))

print(f'F2 word list ready:')
print(f'  Hyland attitude markers (DE) : {len(HYLAND_ATTITUDE_DE)} terms')
print(f'  SentiWS high-intensity       : {len(SENTIWS_HIGH)} terms')
print(f'  Combined (deduplicated)      : {len(AFFECTIVE_MARKERS_DE)} terms')

In [ ]:
# ── F3: Weasel Word Density ────────────────────────────────────────────────────
# Source (lexical): Wikipedia "Words to Watch" — German functional equivalents
# No equivalent DE Wikipedia word list exists (de.wikipedia.org/wiki/
# Wikipedia:Wie_schreibe_ich_gute_Artikel)
# Source (morphological): Konjunktiv I (Mood=Sub, Tense=Pres) as indirect
# speech marker per Helbig & Buscha (1991, p. 194)
# Known limitation: Konjunktiv I = Indikativ for regular verbs in 1st/3rd
# person plural — systematic conservative undercount

WEASEL_WORDS_DE = [
    # Category 1: Vague attribution phrases
    # Source: Wikipedia "Words to Watch" vague attribution patterns
    'einige menschen', 'manche menschen', 'viele menschen',
    'experten sagen', 'experten behaupten',
    'es heißt', 'es wird gesagt', 'es wird behauptet',
    'es wurde vorgeschlagen', 'es wird oft gesagt',
    'es wird berichtet', 'es wird argumentiert', 'es wird angenommen',
    'forscher sagen', 'wissenschaftler sagen', 'historiker sagen',
    'kritiker sagen', 'studien zeigen', 'forschungen zeigen',
    'laut einigen', 'einige argumentieren', 'einige glauben',
    'es gilt als', 'man nimmt an',
    # Note: 'angeblich', 'vorgeblich' removed — already in F1 HEDGES_DE (Klasse 6)

    # Category 2: Numerically vague quantifiers
    # Source: Wikipedia "Words to Watch" vague quantifier patterns
    'viele', 'die meisten', 'mehrere', 'eine reihe von', 'eine mehrheit von',
    'zahlreich', 'zahlreiche', 'zahlreichen', 'zahlreicher', 'zahlreiches',
    'verschieden', 'verschiedene', 'verschiedenen', 'verschiedener',
    'bestimmte', 'einige', 'wenige',

    # Category 3: Presupposition-laden evaluatives
    # Source: Wikipedia "Words to Watch" puffery and evaluative patterns
    'natürlich',          # naturally
    'selbstverständlich', # of course
    'offensichtlich',     # obviously
    'eindeutig',          # clearly
    'zweifellos',         # undoubtedly
    'es versteht sich von selbst',  # it goes without saying
    'unnötig zu sagen',             # needless to say
    'es ist wichtig zu beachten',   # it is important to note
    'es ist erwähnenswert',         # it is worth noting
    'interessanterweise',           # interestingly
    'bemerkenswerterweise',         # notably
    'bedeutend', 'signifikant',     # significantly (replaced 'bedeutsam')
    'legendär',                     # legendary
    'bahnbrechend',                 # groundbreaking
    'revolutionär',                 # revolutionary
    'weltklasse',                   # world-class
    'einzigartig',                  # unique
    'innovativ',                    # innovative
]

def f3_weasel_density(text):
    """
    F3: Weasel word density for German.
    Combines two components:
    (1) Lexical: matching against WEASEL_WORDS_DE
        Source: Wikipedia "Words to Watch" — German functional equivalents
    (2) Morphological: Konjunktiv I detection via spaCy MORPH tags
        (Mood=Sub, Tense=Pres) as indirect speech marker
        Source: Helbig & Buscha (1991, p. 194)
    Known limitation: Konjunktiv I is homophonous with Indikativ for
    regular verbs in 1st/3rd person plural (e.g. wir gehen, sie haben)
    — spaCy defaults to Indikativ → systematic conservative undercount
    """
    if not text or not text.strip():
        return 0.0
    t = text.lower()
    n = max(len(t.split()), 1)

    # Component 1: lexical matching
    count = _density_count(t, WEASEL_WORDS_DE)

    # Component 2: Konjunktiv I morphological detection
    doc = nlp(text)
    for token in doc:
        if (token.pos_ == 'VERB'
                and 'Mood=Sub' in token.morph.to_str()
                and 'Tense=Pres' in token.morph.to_str()):
            count += 1

    return round(count / n * 1000, 4)

print('F3 function ready (lexical + Konjunktiv I morphological)')

In [ ]:
# ── F4: Attribution Verb Bias Ratio ───────────────────────────────────────────
# Source (classification): Recasens et al. (2013) operationalising
# Hooper (1975) assertive predicate distinction
# Source (DE verb list): OWID Kommunikationsverben assertive vs. general
# communication verb distinction (Müller-Spitzer & Proost 2013; Harras et al. 2004)
# Nominalised forms included: German systematically uses nominalisations for attribution

NEUTRAL_REPORT_DE = {
    # Verb lemmas
    'sagen', 'berichten', 'erklären', 'beschreiben', 'angeben',
    'mitteilen', 'erwähnen', 'bemerken','schreiben',
    # Nominalised forms
    'bericht', 'erklärung', 'beschreibung', 'mitteilung',
    'angabe', 'erwähnung', 'bemerkung',
}

BIASED_REPORT_DE = {
    # Verb lemmas — Representatives.Assertives per OWID + Hooper/Recasens
    'behaupten', 'beschuldigen', 'vorwerfen',
    'insistieren', 'anklagen',
    # Nominalised forms
    'behauptung', 'beschuldigung', 'vorwurf', 'anklage',
}

# ── F6: Contrastive Transition Density ────────────────────────────────────────
# Source: Helbig & Buscha (1991):
#   §8.3 (p. 451) Adversativ coordinating: aber, allein, doch,
#                jedoch, sondern; adversative subordinating: während (2nd sense)
#   §4.2.3.1 (p. 341) Konjunktionaladverb: allerdings
#   §19.6 (p. 695) Adversativsatz adverbials: jedoch,
#                 im Gegensatz dazu, demgegenüber
# Predicted direction: S > C (updated from C > S based on EN empirical results
# and Hyland 2018, p. 123: stable texts show higher transition marker density)

CONTRASTIVE_TRANSITIONS_DE = [
    # Helbig & Buscha (1991, p. 451) §8.3 Adversativ, coordinating
    'aber', 'allein', 'doch', 'jedoch', 'sondern',
    # Helbig & Buscha (1991, p. 451) §8.3 Adversativ, subordinating
    'während',       # 2nd (adversative) sense
    # Helbig & Buscha (1991, p. 341) §4.2.3.1 Konjunktionaladverb
    'allerdings',
    # Helbig & Buscha (1991, p. 695) §19.6 Adversativsatz adverbials
    'im Gegensatz dazu', 'demgegenüber',
    # Additional high-frequency adversative connectives
    'dennoch', 'hingegen', 'wohingegen', 'dagegen',
]

# ── F5: Lexical presupposition density (LLM) ──────────────────────────────────
# mDeBERTa-v3-base-xnli-multilingual-nli-2mil7 (Laurer et al. 2022)
# hypothesis_template: {} filled by candidate_labels=['biased', 'neutral']
# Targets lexical-level presupposition/entailment bias (Recasens et al. 2013):
# sentences where word/phrase choice presupposes or entails a one-sided stance
# towards a contested event, person, or claim.
ENTAIL_HYPOTHESIS = (
    "This sentence contains a word or phrase whose meaning "
    "presupposes or entails a one-sided stance towards a "
    "contested event, person, or claim. The sentence is {}."
)

print('Word lists ready')
print(f'  F1 Hedges (DE)           : {len(HEDGES_DE)} terms')
print(f'  F2 Affective Markers (DE): {len(AFFECTIVE_MARKERS_DE)} terms')
print(f'  F3 Weasel Words (DE)     : {len(WEASEL_WORDS_DE)} terms')
print(f'  F4 Biased Report (DE)    : {len(BIASED_REPORT_DE)} / Neutral: {len(NEUTRAL_REPORT_DE)}')
print(f'  F6 Contrastive Trans (DE): {len(CONTRASTIVE_TRANSITIONS_DE)} terms')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FEATURE FUNCTIONS (F1–F4, F6, F7 — lexical/statistical, no LLM)
# ══════════════════════════════════════════════════════════════════════════════

def _density_count(text_lower, wordlist):
    """
    Count wordlist matches with word boundary protection for single words.
    Multi-word phrases matched by substring; single words use regex boundaries.
    """
    count = 0
    for w in wordlist:
        if ' ' in w:
            count += text_lower.count(w.lower())
        else:
            count += len(re.findall(r'\b' + re.escape(w.lower()) + r'\b', text_lower))
    return count

def _density(text, wordlist):
    if not text or not text.strip():
        return 0.0
    t = text.lower()
    n = max(len(t.split()), 1)
    return round(_density_count(t, wordlist) / n * 1000, 4)

# F1: Epistemic Hedge Density
def f1_hedge_density(text):
    """
    F1: Epistemic hedge density for German.
    Combines: (1) lexical matching against HEDGES_DE,
              (2) Konjunktiv II modal verbs via spaCy MORPH tags
                  (Mood=Sub, Tense=Past) restricted to KONJUNKTIV_II_MODALS.
    Source: Helbig & Buscha (1991, p. 507); Diewald (1999, pp. 17–18).
    """
    if not text or not text.strip():
        return 0.0
    t = text.lower()
    n = max(len(t.split()), 1)
    # Lexical count
    count = _density_count(t, HEDGES_DE)
    # Morphological count: Konjunktiv II modal verbs
    doc = nlp(text)
    for token in doc:
        if (token.lemma_.lower() in KONJUNKTIV_II_MODALS
                and 'Mood=Sub' in token.morph.to_str()
                and 'Tense=Past' in token.morph.to_str()):
            count += 1
    return round(count / n * 1000, 4)

# F2: Affective Stance Marker Density
def f2_affective_density(text):
    """
    F2: Affective stance marker density for German.
    AFFECTIVE_MARKERS_DE = Hyland (2018) attitude markers (DE equivalents)
    UNION SentiWS v2.0 (Remus et al. 2010), |weight| >= 0.5.
    Source: Hyland (2018, pp. 268–269); Remus et al. (2010).
    """
    return _density(text, AFFECTIVE_MARKERS_DE)

# F3: Weasel Word Density
def f3_weasel_density(text):
    """
    F3: Weasel word density for German.
    Combines: (1) lexical matching against WEASEL_WORDS_DE,
              (2) Konjunktiv I detection via spaCy MORPH tags
                  (Mood=Sub, Tense=Pres) as indirect speech marker.
    Source: Wikipedia Words to Watch (DE equivalents);
            Helbig & Buscha (1991, p. 194).
    Limitation: Konjunktiv I = Indikativ for regular verbs in
    1st/3rd person plural — conservative systematic undercount.
    """
    if not text or not text.strip():
        return 0.0
    t = text.lower()
    n = max(len(t.split()), 1)
    count = _density_count(t, WEASEL_WORDS_DE)
    # Konjunktiv I morphological count
    doc = nlp(text)
    for token in doc:
        if (token.pos_ == 'VERB'
                and 'Mood=Sub' in token.morph.to_str()
                and 'Tense=Pres' in token.morph.to_str()):
            count += 1
    return round(count / n * 1000, 4)

# F4: Attribution Verb Bias Ratio
def f4_attribution_bias(text):
    """
    Ratio of biased to total report verbs/nominalisations via spaCy lemmatisation.
    Covers verb lemmas (POS=VERB) and nominalised forms (POS=NOUN).
    Returns NaN if total count < 3.
    Source: Recasens et al. (2013); Müller-Spitzer & Proost (2013).
    """
    if not text or not text.strip():
        return np.nan
    doc = nlp(text)  # original case for accurate lemmatisation
    biased  = 0
    neutral = 0
    for token in doc:
        lemma = token.lemma_.lower()
        if token.pos_ in ('VERB', 'NOUN'):
            if lemma in BIASED_REPORT_DE:
                biased += 1
            elif lemma in NEUTRAL_REPORT_DE:
                neutral += 1
    total = biased + neutral
    if total < 3:
        return np.nan
    return round(biased / total, 4)

# F6: Contrastive Transition Density
def f6_contrastive_density(text):
    """
    F6: Contrastive transition density for German.
    Source: Helbig & Buscha (1991): §8.3 (p. 451) adversative
    coordinating/subordinating conjunctions; §4.2.3.1 (p. 341)
    Konjunktionaladverb allerdings; §19.6 (p. 695) adverbial connectives.
    Predicted direction: S > C.
    """
    return _density(text, CONTRASTIVE_TRANSITIONS_DE)

# F7: Lexical Diversity (MTLD)
def f7_mtld(text):
    """
    F7: Lexical diversity via MTLD (McCarthy & Jarvis 2010).
    Language-agnostic algorithm; threshold=0.72; minimum 50 tokens.
    Source: lexicalrichness library.
    """
    if not text or len(text.split()) < 50:
        return np.nan
    try:
        return round(LexicalRichness(text).mtld(threshold=0.72), 4)
    except:
        return np.nan

# Hilfsfunktion: klassifizierbare Sätze filtern (DE-Version)
def _is_classifiable(sent_text):
    """Filter sentences that cannot carry epistemological bias — DE version."""
    tokens = sent_text.strip().split()
    if len(tokens) < 6:
        return False
    doc = nlp(sent_text)
    if not any(t.pos_ == 'VERB' for t in doc):
        return False
    num_ratio = sum(1 for t in doc if t.like_num or t.pos_ == 'NUM') / max(len(tokens), 1)
    if num_ratio > 0.4:
        return False
    return True

print('Feature functions F1–F4, F6, F7 ready')

## 3b. LLM Feature (F5) — GPU. With Checkpointing.

In [ ]:
# ── LLM-Modell laden ──────────────────────────────────────────────────────────
device = 0 if torch.cuda.is_available() else -1
print(f'Device: {"GPU" if device == 0 else "CPU"}')

# F5: Multilingual NLI for lexical presupposition density
# mDeBERTa is multilingual; German supported natively
print('Loading F5 NLI model...')
nli_pipe = pipeline(
    'zero-shot-classification',
    model='MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7',
    device=device,
)
print('NLI model loaded')

In [ ]:
# ── F5: Lexical presupposition density (LLM) ────────────────────────────────────
# hypothesis_template: {} filled by candidate_labels=['biased', 'neutral']
# Targets lexical-level presupposition/entailment bias (Recasens et al. 2013):
# sentences where word/phrase choice presupposes or entails a one-sided stance
# towards a contested event, person, or claim.
ENTAIL_HYPOTHESIS = (
    "This sentence contains a word or phrase whose meaning "
    "presupposes or entails a one-sided stance towards a "
    "contested event, person, or claim. The sentence is {}."
)

def _nli_density(text, hypothesis, batch_size=32):
    """
    Proportion of classifiable sentences per article where NLI model
    assigns entailment score > 0.5 to the given hypothesis.
    Denominator: classifiable sentences only (consistent with EN version).
    Returns NaN for empty or fully unclassifiable texts.
    """
    if not text or not text.strip():
        return np.nan
    doc   = nlp(text)
    sents = [s.text.strip() for s in doc.sents if _is_classifiable(s.text)]
    if not sents:
        return np.nan
    sents_trunc = [s[:512] for s in sents]
    results = nli_pipe(
        sents_trunc,
        candidate_labels=['biased', 'neutral'],
        hypothesis_template=hypothesis,
        batch_size=batch_size,
        multi_label=False,
    )
    biased_count = sum(
        1 for r in results
        if r['labels'][0] == 'biased' and r['scores'][0] > 0.5
    )
    return round(biased_count / len(sents), 4)

def f5_entailment_density(text):
    """
    F5: Lexical framing bias density.
    Proportion of classifiable sentences where a word or phrase
    presupposes or entails a one-sided stance towards a contested
    event, person, or claim, via zero-shot NLI.
    Source: Laurer et al. (2022); Recasens et al. (2013).
    """
    return _nli_density(text, ENTAIL_HYPOTHESIS)

print('F5 function ready')

## 4. Feature Extraction

In [ ]:
# ── Checkpoint helpers ─────────────────────────────────────────────────────────

def load_checkpoint(path):
    if os.path.exists(path):
        df_done = pd.read_csv(path)
        done_ids = set(df_done['title'])
        print(f'  Checkpoint loaded: {len(done_ids)} articles already done')
        return df_done, done_ids
    return pd.DataFrame(), set()

def append_checkpoint(records, path):
    pd.DataFrame(records).to_csv(
        path,
        mode='a',
        header=not os.path.exists(path),
        index=False,
    )

print('Checkpoint helpers ready')

In [ ]:
# ── Run F5 with checkpoint ─────────────────────────────────────────────────────
# NOTE: if ENTAIL_HYPOTHESIS was changed, delete F5_CHECKPOINT before running.
if len(pairs) == 0:
    print('⚠️  No pairs loaded — F5 checkpoint skipped (placeholder mode)')
    df_f5 = pd.DataFrame(columns=['title', 'f5_entailment'])
else:
    all_articles = [(rec['title'], rec['text']) for _, rec in df.iterrows()]
    _, f5_done_ids = load_checkpoint(F5_CHECKPOINT)
    f5_buffer = []
    total_done = len(f5_done_ids)
    print(f'Running F5 on {len(all_articles)} articles ({total_done} already done)...')
    for title, text in all_articles:
        if title in f5_done_ids:
            continue
        score = f5_entailment_density(text)
        f5_buffer.append({'title': title, 'f5_entailment': score})
        if len(f5_buffer) % 50 == 0:
            append_checkpoint(f5_buffer, F5_CHECKPOINT)
            total_done += len(f5_buffer)
            print(f'  F5 checkpoint: {total_done}/{len(all_articles)} done')
            f5_buffer = []
    if f5_buffer:
        append_checkpoint(f5_buffer, F5_CHECKPOINT)
        total_done += len(f5_buffer)
    df_f5 = pd.read_csv(F5_CHECKPOINT)
    print(f'F5 complete: {len(df_f5)} articles')

In [ ]:
# ── Build full feature matrix ──────────────────────────────────────────────────
# F5 from checkpoint; all others computed inline

if len(pairs) == 0:
    print('⚠️  No pairs loaded — feature matrix in placeholder mode (empty)')
    df = pd.DataFrame(columns=['title', 'label', 'label_name', 'topic',
                                'word_count', 'age_days', 'pair_id',
                                'f1_hedge', 'f2_affective', 'f3_weasel',
                                'f4_attrib_bias', 'f5_entailment',
                                'f6_contrastive', 'f7_mtld'])
else:
    f5_lookup = (df_f5.drop_duplicates(subset='title', keep='last')
             .set_index('title')['f5_entailment']
             .to_dict()) if not df_f5.empty else {}

    records = []
    total   = len(pairs) * 2
    done    = 0

    print(f'Extracting features for {len(pairs)} pairs ({total} articles)...')
    print('(spaCy features ~1 sec/article)')

    for p in pairs:
        for role, art in [('contested', p['contested']), ('stable', p['stable'])]:
            title = art['title']
            text  = art.get('clean_text', '')

            record = {
                'title'          : title,
                'label'          : 0 if role == 'contested' else 1,
                'label_name'     : role,
                'topic'          : art.get('topic', 'other'),
                'word_count'     : art.get('word_count', 0),
                'age_days'       : art.get('age_days', 0),
                'pair_id'        : p['contested']['title'],
                # Layer 1
                'f1_hedge'       : f1_hedge_density(text),
                'f2_affective'   : f2_affective_density(text),
                # Layer 2
                'f3_weasel'      : f3_weasel_density(text),
                'f4_attrib_bias' : f4_attribution_bias(text),
                'f5_entailment'  : f5_lookup.get(title, np.nan),
                # Layer 3
                'f6_contrastive' : f6_contrastive_density(text),
                'f7_mtld'        : f7_mtld(text),
            }
            records.append(record)
            done += 1
            if done % 20 == 0:
                print(f'  {done}/{total} done...')

    df = pd.DataFrame(records)
    df.to_csv('features_v4_de.csv', index=False)
    print(f'\nDone. Feature matrix: {df.shape[0]} rows x {df.shape[1]} cols')
    print(f'  Contested : {(df.label==0).sum()}')
    print(f'  Stable    : {(df.label==1).sum()}')

## 5. Feature Overview & Direction Check

In [ ]:
LING_FEATURES = ['f1_hedge', 'f2_affective', 'f3_weasel', 'f4_attrib_bias',
                 'f5_entailment', 'f6_contrastive', 'f7_mtld']

LABELS = {
    'f1_hedge'       : 'F1 Epistemic hedges',
    'f2_affective'   : 'F2 Affective stance markers',
    'f3_weasel'      : 'F3 Weasel words',
    'f4_attrib_bias' : 'F4 Attribution bias ratio',
    'f5_entailment'  : 'F5 Lexical presupposition density',
    'f6_contrastive' : 'F6 Contrastive transitions',
    'f7_mtld'        : 'F7 Lexical diversity (MTLD)',
}

LAYER_MAP = {
    'f1_hedge'       : 'L1',
    'f2_affective'   : 'L1',
    'f3_weasel'      : 'L2',
    'f4_attrib_bias' : 'L2',
    'f5_entailment'  : 'L2',
    'f6_contrastive' : 'L3',
    'f7_mtld'        : 'L3',
}

PREDICTED = {
    'f1_hedge'       : +1,   # C > S
    'f2_affective'   : +1,   # C > S
    'f3_weasel'      : +1,   # C > S
    'f4_attrib_bias' : +1,   # C > S
    'f5_entailment'  : +1,   # C > S
    'f6_contrastive' : -1,   # S > C (aktualisiert von EN Ergebnissen + Hyland 2018)
    'f7_mtld'        : +1,   # C > S
}

if len(df) == 0:
    print('⚠️  DataFrame empty — direction check skipped (placeholder mode)')
else:
    c_df = df[df.label == 0]
    s_df = df[df.label == 1]

    print(f'{"Feature":<35} {"Contested":>10} {"Stable":>10} {"Diff%":>8}  {"Dir":>5}  Layer')
    print('-' * 80)
    n_correct = 0
    for f in LING_FEATURES:
        if f not in df.columns: continue
        cm  = c_df[f].mean()
        sm  = s_df[f].mean()
        diff = (cm - sm) / (abs(sm) + 1e-9) * 100
        ok   = '✓' if ((cm > sm) == (PREDICTED[f] == 1)) else '✗'
        if ok == '✓': n_correct += 1
        print(f'{LABELS[f]:<35} {cm:>10.4f} {sm:>10.4f} {diff:>+8.1f}%  {ok:>5}   {LAYER_MAP[f]}')

    print(f'\nDirection correct: {n_correct}/{len(LING_FEATURES)}')

## 6. NaN Report

In [ ]:
if len(df) == 0:
    print('⚠️  DataFrame empty — NaN report skipped')
else:
    nan_counts = df[LING_FEATURES].isna().sum()
    print('NaN counts per feature:')
    for f in LING_FEATURES:
        pct = nan_counts[f] / len(df) * 100
        flag = '  ⚠ >10%' if pct > 10 else ''
        print(f'  {LABELS[f]:<35} {nan_counts[f]:>5} ({pct:.1f}%){flag}')

## 7. VIF Check

In [ ]:
if len(df) == 0:
    print('⚠️  DataFrame empty — VIF check skipped')
else:
    df_vif = df[LING_FEATURES].dropna()
    X_vif  = StandardScaler().fit_transform(df_vif.values)

    vif_vals = [variance_inflation_factor(X_vif, i) for i in range(X_vif.shape[1])]

    print(f'{"Feature":<35} {"VIF":>8}  Status')
    print('-' * 60)
    for f, v in zip(LING_FEATURES, vif_vals):
        status = 'OK' if v < 5 else ('WARN (consider merge)' if v < 10 else 'HIGH — must merge')
        print(f'{LABELS[f]:<35} {v:>8.2f}  {status}')

    print('\nRule: VIF < 5 keep | 5–10 consider merging | >10 must merge')

## 8. Wilcoxon Signed-Rank Tests on Matched Pairs

In [ ]:
if len(pairs) == 0:
    print('⚠️  No pairs loaded — Wilcoxon skipped')
    raw_results = []
    diff_df = pd.DataFrame()
else:
    diff_records = []
    for p in pairs:
        c_title = p['contested']['title']
        s_title = p['stable']['title']
        c_row   = df[df.title == c_title]
        s_row   = df[df.title == s_title]
        if c_row.empty or s_row.empty: continue
        c_row, s_row = c_row.iloc[0], s_row.iloc[0]
        rec = {f: c_row[f] - s_row[f] for f in LING_FEATURES if f in df.columns}
        rec['pair_id'] = c_title
        diff_records.append(rec)

    diff_df = pd.DataFrame(diff_records)
    print(f'Valid pairs for Wilcoxon: {len(diff_df)}\n')

    raw_results = []
    for f in LING_FEATURES:
        if f not in diff_df.columns: continue
        vals = diff_df[f].dropna()
        if len(vals) < 5: continue
        stat, p_val = wilcoxon(vals)
        n      = len(vals)
        mean_w = n * (n + 1) / 4
        std_w  = np.sqrt(n * (n + 1) * (2 * n + 1) / 24)
        z      = (stat - mean_w) / std_w
        r_rb   = round(abs(z) / np.sqrt(n), 3)
        raw_results.append({
            'feature'   : f,
            'mean_diff' : vals.mean(),
            'pct_c_hi'  : (vals > 0).mean() * 100,
            'p_raw'     : p_val,
            'r_rb'      : r_rb,
            'n'         : len(vals),
        })

    if raw_results:
        p_vals  = [r['p_raw'] for r in raw_results]
        _, p_holm, _, _ = multipletests(p_vals, method='holm')
        for r, p_h in zip(raw_results, p_holm):
            r['p_holm'] = p_h
            r['sig']    = '***' if p_h < 0.001 else ('**' if p_h < 0.01 else ('*' if p_h < 0.05 else ('~' if p_h < 0.10 else 'ns')))

        print(f'{"Feature":<35} {"Mean Δ":>9} {"C>S%":>7} {"p (Holm)":>10}  Sig   Layer')
        print('-' * 80)
        for r in raw_results:
            f = r['feature']
            print(f'{LABELS[f]:<35} {r["mean_diff"]:>+9.4f} {r["pct_c_hi"]:>6.1f}% {r["p_holm"]:>10.4f}  {r["sig"]:<5} {LAYER_MAP[f]}')

        n_sig = sum(1 for r in raw_results if r['p_holm'] < 0.05)
        print(f'\n* p<.05  ** p<.01  *** p<.001  ~ p<.10  ns = not significant (Holm-corrected)')
        print(f'Significant features: {n_sig}/{len(LING_FEATURES)}')

## 9. Logistic Regression with Topic Controls

In [ ]:
if len(df) == 0:
    print('⚠️  DataFrame empty — logistic regression skipped')
    results = {}
    topic_cols = []
    df_model = df.copy()
else:
    df_topic   = pd.get_dummies(df['topic'], prefix='topic', drop_first=True)
    df_model   = pd.concat([df.reset_index(drop=True), df_topic.reset_index(drop=True)], axis=1)
    topic_cols = list(df_topic.columns)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

    feature_sets = {
        'L1 only (Stance)'              : [f for f in LING_FEATURES if LAYER_MAP[f] == 'L1'],
        'L2 only (Attribution/Framing)' : [f for f in LING_FEATURES if LAYER_MAP[f] == 'L2'],
        'L3 only (Discourse)'           : [f for f in LING_FEATURES if LAYER_MAP[f] == 'L3'],
        'All 7 linguistic'              : LING_FEATURES,
        'All 7 + topic controls'        : LING_FEATURES + topic_cols,
    }

    print(f'{"Feature-Set":<35} {"F1":>8} {"Acc":>8} {"vs Baseline":>12}')
    print('-' * 67)

    results = {}
    for name, feats in feature_sets.items():
        avail = [f for f in feats if f in df_model.columns]
        d     = df_model[avail + ['label']].dropna()
        if len(d) < 20 or len(avail) < 1: continue
        X    = StandardScaler().fit_transform(d[avail].values)
        y    = d['label'].values
        base = max(y.mean(), 1 - y.mean())
        f1s  = cross_val_score(lr, X, y, cv=cv, scoring='f1_macro')
        accs = cross_val_score(lr, X, y, cv=cv, scoring='accuracy')
        results[name] = {'f1': f1s.mean(), 'acc': accs.mean(), 'f1_std': f1s.std()}
        print(f'{name:<35} {f1s.mean():>8.3f} {accs.mean():>8.3f} {accs.mean()-base:>+12.3f}')

    print(f'\nMajority-class baseline: {max(df.label.mean(), 1-df.label.mean()):.3f}')

## 10. Bootstrap 95% CI

In [ ]:
if len(df) == 0:
    print('⚠️  DataFrame empty — bootstrap skipped')
    best_feats = LING_FEATURES
    coef_arr = np.array([])
    ling_idx = []
else:
    lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    best_feats = [f for f in LING_FEATURES if f in df_model.columns] + topic_cols
    d_full     = df_model[best_feats + ['label']].dropna()
    X_full     = StandardScaler().fit_transform(d_full[best_feats].values)
    y_full     = d_full['label'].values

    lr.fit(X_full, y_full)

    coef_boot = []
    for _ in range(1000):
        X_b, y_b = resample(X_full, y_full, random_state=None)
        if len(set(y_b)) < 2: continue
        m = LogisticRegression(class_weight='balanced', max_iter=1000)
        m.fit(X_b, y_b)
        coef_boot.append(m.coef_[0])

    coef_arr  = np.array(coef_boot)
    ling_idx  = [i for i, f in enumerate(best_feats) if f in LING_FEATURES]

    print(f'{"Feature":<35} {"Koef":>8} {"95% KI":>22}  Stable?  Layer')
    print('-' * 80)

    stable_feats, unstable_feats = [], []
    for i in ling_idx:
        f  = best_feats[i]
        lo = np.percentile(coef_arr[:, i], 2.5)
        hi = np.percentile(coef_arr[:, i], 97.5)
        c  = lr.coef_[0][i]
        ok = 'YES' if (lo > 0 or hi < 0) else 'unstable'
        if ok == 'YES': stable_feats.append(f)
        else: unstable_feats.append(f)
        print(f'{LABELS.get(f,f):<35} {c:>+8.3f} [{lo:>+7.3f}, {hi:>+7.3f}]  {ok:<8} {LAYER_MAP.get(f,"")}')

    print(f'\nStable CIs   : {[LABELS[f] for f in stable_feats]}')
    print(f'Unstable CIs : {[LABELS[f] for f in unstable_feats]}')

## 11. Visualisations

In [ ]:
# ── Coefficient Plot ──────────────────────────────────────────────────────────
LAYER_COLORS = {'L1': '#E05C3A', 'L2': '#5B8FD4', 'L3': '#4CAF82'}
LAYER_NAMES  = {'L1': 'L1 Stance', 'L2': 'L2 Attribution', 'L3': 'L3 Organisation'}

if len(coef_arr) == 0 or len(ling_idx) == 0:
    print('⚠️  No coefficients available — coefficient plot skipped')
else:
    coef_data = []
    for i in ling_idx:
        f  = best_feats[i]
        lo = np.percentile(coef_arr[:, i], 2.5)
        hi = np.percentile(coef_arr[:, i], 97.5)
        c  = lr.coef_[0][i]
        coef_data.append({
            'label': LABELS[f],
            'coef' : c,
            'lo'   : lo,
            'hi'   : hi,
            'layer': LAYER_MAP[f],
        })
    coef_data.sort(key=lambda x: x['coef'])

    fig, ax = plt.subplots(figsize=(10, 6))
    for j, d in enumerate(coef_data):
        color = LAYER_COLORS[d['layer']]
        ax.barh(j, d['coef'], color=color, alpha=0.8, height=0.6)
        ax.plot([d['lo'], d['hi']], [j, j], color='black', linewidth=1.5)
        ax.plot([d['lo'], d['lo']], [j-0.15, j+0.15], color='black', linewidth=1.5)
        ax.plot([d['hi'], d['hi']], [j-0.15, j+0.15], color='black', linewidth=1.5)

    ax.set_yticks(range(len(coef_data)))
    ax.set_yticklabels([d['label'] for d in coef_data], fontsize=10)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel(
        'Standardised coefficient (← predicts Contested | predicts Stable →)',
        fontsize=10
    )
    ax.set_title(
        'Logistic Regression Coefficients — 7 Linguistic Features (DE)',
        fontsize=12, fontweight='bold'
    )
    legend_patches = [mpatches.Patch(color=LAYER_COLORS[l], label=LAYER_NAMES[l])
                      for l in LAYER_COLORS]
    ax.legend(handles=legend_patches, loc='lower right', fontsize=10)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('coefficients_v4_de.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: coefficients_v4_de.png')

In [ ]:
# ── Violinplots (7 Panels) ────────────────────────────────────────────────────
if len(df) == 0:
    print('⚠️  DataFrame empty — violin plots skipped')
else:
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    axes_flat = axes.flatten()

    for ax, f in zip(axes_flat, LING_FEATURES):
        c_vals = df[df.label == 0][f].dropna()
        s_vals = df[df.label == 1][f].dropna()
        if len(c_vals) == 0 or len(s_vals) == 0:
            ax.set_visible(False)
            continue
        parts  = ax.violinplot([c_vals.values, s_vals.values],
                                positions=[0, 1], showmedians=True, showextrema=False)
        for body, color in zip(parts['bodies'], ['#E05C3A', '#5B8FD4']):
            body.set_facecolor(color)
            body.set_alpha(0.7)
        parts['cmedians'].set_color('black')
        parts['cmedians'].set_linewidth(2)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['C', 'S'], fontsize=10)
        ax.set_title(LABELS[f], fontsize=8, fontweight='bold')
        ax.set_xlabel(LAYER_MAP[f], fontsize=8, color='gray')
        ax.grid(axis='y', alpha=0.3)

    # Hide unused panel
    axes_flat[-1].set_visible(False)

    fig.suptitle('Feature Distributions: Contested (C) vs Stable (S) — DE', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('distributions_v4_de.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: distributions_v4_de.png')

In [ ]:
# ── Spearman-Korrelationsmatrix ────────────────────────────────────────────────
if len(df) == 0:
    print('⚠️  DataFrame empty — correlation matrix skipped')
else:
    n_f  = len(LING_FEATURES)
    corr = np.zeros((n_f, n_f))
    for i, f1 in enumerate(LING_FEATURES):
        for j, f2 in enumerate(LING_FEATURES):
            vals = df[[f1, f2]].dropna()
            if len(vals) > 5:
                result = stats.spearmanr(vals[f1], vals[f2])
                r = float(result[0])
                corr[i,j] = r
            else:
                corr[i,j] = np.nan

    fig, ax = plt.subplots(figsize=(10, 9))
    im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
    plt.colorbar(im, ax=ax, label='Spearman r')
    tick_labels = [LABELS[f] for f in LING_FEATURES]
    ax.set_xticks(range(n_f))
    ax.set_yticks(range(n_f))
    ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(tick_labels, fontsize=8)
    for i in range(n_f):
        for j in range(n_f):
            if not np.isnan(corr[i,j]):
                ax.text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center',
                        fontsize=7, color='white' if abs(corr[i,j]) > 0.5 else 'black')
    ax.set_title('Feature Correlation Matrix (Spearman) — DE', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('correlation_matrix_v4_de.png', dpi=150, bbox_inches='tight')
    plt.show()

    max_off = np.nanmax(np.abs(corr - np.eye(n_f)))
    print(f'Max off-diagonal |r| = {max_off:.3f}')
    print(f'Layers empirically separable (all |r| < 0.5): {max_off < 0.5}')

    cross_layer_pairs = [
        (i, j) for i in range(n_f) for j in range(n_f)
        if i != j and LAYER_MAP[LING_FEATURES[i]] != LAYER_MAP[LING_FEATURES[j]]
    ]
    max_cross = max(abs(corr[i, j]) for i, j in cross_layer_pairs if not np.isnan(corr[i, j]))
    print(f'Max cross-layer |r|  = {max_cross:.3f}')
    print(f'Layers empirically separable (cross-layer |r| < 0.5): {max_cross < 0.5}')

## 12. Summary

In [ ]:
print('=' * 80)
print('RESULTS SUMMARY — DE (v4, 7 features)')
print('=' * 80)
print(f'\nMatched pairs: {len(pairs)}')

if raw_results:
    print(f'\n{"Feature":<35} {"Dir":>5} {"Wilcoxon":>10} {"Boot CI":>10}  Layer')
    print('-' * 70)
    for r in raw_results:
        f     = r['feature']
        if f not in best_feats or len(coef_arr) == 0:
            continue
        i = best_feats.index(f)
        if i >= coef_arr.shape[1]:
            continue
        lo    = np.percentile(coef_arr[:, i], 2.5)
        hi    = np.percentile(coef_arr[:, i], 97.5)
        ci_ok = 'stabil' if (lo > 0 or hi < 0) else 'unstable'
        cm    = df[df.label==0][f].mean() if len(df) > 0 else np.nan
        sm    = df[df.label==1][f].mean() if len(df) > 0 else np.nan
        dir_ok = '✓' if (not np.isnan(cm) and not np.isnan(sm) and (cm > sm) == (PREDICTED[f] == 1)) else '✗'
        print(f'{LABELS[f]:<35} {dir_ok:>5} {r["sig"]:>10} {ci_ok:>10}  {LAYER_MAP[f]}')

best_name = 'All 7 + topic controls'
if best_name in results:
    r = results[best_name]
    print(f'\nLogistische Regression (all 7 + topic controls):')
    print(f'  F1 (macro) : {r["f1"]:.3f} ± {r["f1_std"]:.3f}')
    print(f'  Accuracy   : {r["acc"]:.3f}')
    print(f'  Baseline   : {max(df.label.mean(), 1-df.label.mean()):.3f}' if len(df) > 0 else '')

n_sig = sum(1 for r in raw_results if r.get('p_holm', 1) < 0.05)
print(f'\nTotal significant (Wilcoxon Holm p<.05): {n_sig}/{len(LING_FEATURES)}')

## 13. Permutation Test

In [ ]:
if len(diff_df) == 0:
    print('⚠️  No differences available — permutation test skipped')
elif not raw_results:
    print('⚠️  No Wilcoxon results available — permutation test skipped')
else:
    n_permutations = 1000
    perm_sig_counts = []

    for _ in range(n_permutations):
        perm_diffs = diff_df[LING_FEATURES].copy()
        for idx in perm_diffs.index:
            if np.random.rand() > 0.5:
                perm_diffs.loc[idx] = -perm_diffs.loc[idx]

        p_vals_perm = []
        for f in LING_FEATURES:
            if f not in perm_diffs.columns: continue
            vals = perm_diffs[f].dropna()
            if len(vals) < 5: continue
            _, p = wilcoxon(vals)
            p_vals_perm.append(p)
        if p_vals_perm:
            _, p_holm_perm, _, _ = multipletests(p_vals_perm, method='holm')
            count = sum(p < 0.05 for p in p_holm_perm)
        else:
            count = 0
        perm_sig_counts.append(count)

    real_sig = n_sig
    p_perm = np.mean(np.array(perm_sig_counts) >= real_sig)
    print(f'Observed significant features : {real_sig}/{len(LING_FEATURES)}')
    print(f'Permutation mean under null   : {np.mean(perm_sig_counts):.2f}')
    print(f'Permutation p-value           : {p_perm:.4f}')

## 14. AUC (5-Fold CV)

In [ ]:
from sklearn.metrics import roc_auc_score

if len(df) == 0:
    print('⚠️  DataFrame empty — AUC skipped')
else:
    d = df_model[LING_FEATURES + ['label']].dropna()
    if len(d) < 10:
        print('⚠️  Insufficient data for AUC computation')
    else:
        X = StandardScaler().fit_transform(d[LING_FEATURES].values)
        y = d['label'].values
        lr_auc = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
        cv_auc = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        aucs   = cross_val_score(lr_auc, X, y, cv=cv_auc, scoring='roc_auc')
        accs   = cross_val_score(lr_auc, X, y, cv=cv_auc, scoring='accuracy')
        print(f'AUC      : {aucs.mean():.3f} ± {aucs.std():.3f}')
        print(f'Accuracy : {accs.mean():.3f} ± {accs.std():.3f}')
        print(f'Baseline : {max(y.mean(), 1-y.mean()):.3f}')